In [1]:
#importando as bibliotecas nescessarias 
import pdfplumber
import pandas as pd
import re
import os

- Testando extrair texto de apenas uma pagina para encontrar um certo padrão, como: cabeçalhos, linhas de valores etc...

In [2]:
file = r"Dados_Brutos\_receita_jan2025-consol_cer16400_2271_27023424.pdf"

with pdfplumber.open(file) as pdf:
    page = pdf.pages[0]
    text = page.extract_text()

print(text)

PREF MUN ESTANCIA TURIS DE OLIMPIA
Balancete da Receita Janeiro/2025 CONSOLIDADO
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.1.001 1 IMPOSTOS S/PREDIAL URBANO
Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a
0,00 41.089,58 41.089,58 12.600.000,00 12.600.000,00 -12.558.910,42
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.1.002 2 IMP. S/PROP. TERRITORIAL URBANA
Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a
0,00 43.037,06 43.037,06 7.500.000,00 7.500.000,00 -7.456.962,94
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.2.001 3 IPTU M/J
Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a
0,00 972,96 972,96 48.000,00 48.000,00 -47.027,04
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.3.001 4 IPTU DIV ATIVA
Anterior 

- Fazendo estrutura de registro (record parsing)
- metodo de parsing a ser utilizado: REGEX
- transformar em dados estruturados

In [3]:
# teste de extração texto
import pdfplumber

arquivo = r"Dados_Brutos\_receita_jan2025-consol_cer16400_2271_27023424.pdf"

with pdfplumber.open(arquivo) as pdf:
    texto = ""
    
    for pagina in pdf.pages:
        texto += pagina.extract_text() + "\n"
texto_limpo = " ".join(texto.split())
print(texto[:1000])

PREF MUN ESTANCIA TURIS DE OLIMPIA
Balancete da Receita Janeiro/2025 CONSOLIDADO
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.1.001 1 IMPOSTOS S/PREDIAL URBANO
Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a
0,00 41.089,58 41.089,58 12.600.000,00 12.600.000,00 -12.558.910,42
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.1.002 2 IMP. S/PROP. TERRITORIAL URBANA
Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a
0,00 43.037,06 43.037,06 7.500.000,00 7.500.000,00 -7.456.962,94
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.2.001 3 IPTU M/J
Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a
0,00 972,96 972,96 48.000,00 48.000,00 -47.027,04
Natureza da Receita F i c h a D e s c r i ç ã o
1.1.1.2.50.0.3.001 4 IPTU DIV ATIVA
Anterior 

- separando em linhas

In [4]:
linhas = texto.split("\n")
print(linhas)

['PREF MUN ESTANCIA TURIS DE OLIMPIA', 'Balancete da Receita Janeiro/2025 CONSOLIDADO', 'Natureza da Receita F i c h a D e s c r i ç ã o', '1.1.1.2.50.0.1.001 1 IMPOSTOS S/PREDIAL URBANO', 'Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a', '0,00 41.089,58 41.089,58 12.600.000,00 12.600.000,00 -12.558.910,42', 'Natureza da Receita F i c h a D e s c r i ç ã o', '1.1.1.2.50.0.1.002 2 IMP. S/PROP. TERRITORIAL URBANA', 'Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a', '0,00 43.037,06 43.037,06 7.500.000,00 7.500.000,00 -7.456.962,94', 'Natureza da Receita F i c h a D e s c r i ç ã o', '1.1.1.2.50.0.2.001 3 IPTU M/J', 'Anterior A r r e c a d a d o Mês A r r e c a d a d o A no P r e v i s ã o P r e v i s ã o A t u a lizada D i f e r e n ç a', '0,00 972,96 972,96 48.000,00 48.000,00 -47.027,04', 'Natureza da Receita F i c h a D e s c r i ç ã o',

In [5]:
padrao = r"(\d+\.\d+\.\d+\.\d+\.\d+\.\d+\.\d+\.\d+)\s+\d+\s+([A-ZÀ-Üa-zà-ü\s\/\.\-]+?)\s+([\d\.,\-]+\s+[\d\.,\-]+\s+[\d\.,\-]+\s+[\d\.,\-]+\s+[\d\.,\-]+\s+[\d\.,\-]+)"
matches = re.findall(padrao, texto_limpo)

dados = []

for m in matches:
    valores = re.findall(r"[\d\.,\-]+", m[2])

    dados.append({
        "codigo": m[0],
        "descricao": re.split(r"Anterior|Arrecadado|Mês", m[1])[0].strip(),
        "anterior": valores[0],
        "mes": valores[1],
        "ano": valores[2],
        "previsto": valores[3],
        "atualizado": valores[4],
        "diferenca": valores[5],
    })

df = pd.DataFrame(dados)
print(df.head())

               codigo                        descricao anterior         mes  \
0  1.1.1.2.50.0.1.001        IMPOSTOS S/PREDIAL URBANO     0,00   41.089,58   
1  1.1.1.2.50.0.1.002  IMP. S/PROP. TERRITORIAL URBANA     0,00   43.037,06   
2  1.1.1.2.50.0.2.001                         IPTU M/J     0,00      972,96   
3  1.1.1.2.50.0.3.001                   IPTU DIV ATIVA     0,00  323.935,72   
4  1.1.1.2.50.0.4.001               M/J DIV ATIVA IPTU     0,00   70.176,85   

          ano       previsto     atualizado       diferenca  
0   41.089,58  12.600.000,00  12.600.000,00  -12.558.910,42  
1   43.037,06   7.500.000,00   7.500.000,00   -7.456.962,94  
2      972,96      48.000,00      48.000,00      -47.027,04  
3  323.935,72   3.500.000,00   3.500.000,00   -3.176.064,28  
4   70.176,85     810.000,00     810.000,00     -739.823,15  


- Limpeza basica

In [6]:
def converter(valor):
    return float(valor.replace(".", "").replace(",", "."))

colunas = ["anterior", "mes", "ano", "previsto", "atualizado", "diferenca"]

df[colunas] = df[colunas].apply(lambda col: col.map(converter))

print(df.head())

               codigo                        descricao  anterior        mes  \
0  1.1.1.2.50.0.1.001        IMPOSTOS S/PREDIAL URBANO       0.0   41089.58   
1  1.1.1.2.50.0.1.002  IMP. S/PROP. TERRITORIAL URBANA       0.0   43037.06   
2  1.1.1.2.50.0.2.001                         IPTU M/J       0.0     972.96   
3  1.1.1.2.50.0.3.001                   IPTU DIV ATIVA       0.0  323935.72   
4  1.1.1.2.50.0.4.001               M/J DIV ATIVA IPTU       0.0   70176.85   

         ano    previsto  atualizado    diferenca  
0   41089.58  12600000.0  12600000.0 -12558910.42  
1   43037.06   7500000.0   7500000.0  -7456962.94  
2     972.96     48000.0     48000.0    -47027.04  
3  323935.72   3500000.0   3500000.0  -3176064.28  
4   70176.85    810000.0    810000.0   -739823.15  


- Juntando os 12 pdf em um dataframe unico

In [7]:
#listando os arquivos
pasta = "Dados_Brutos"
arquivos = [f for f in os.listdir(pasta) if f.endswith(".pdf")]
print(arquivos)

['_receita_ago2025-consol_cer16400_9426_29094402.pdf', '_receita_consol_abr2025_cer16400_3863_29025108.pdf', '_receita_consol_fev2025_cer16400_3381_27023618.pdf', '_receita_consol_mar2025_cer16400_6033_29031307.pdf', '_receita_dez2025-consolidado_cer16400_8343_28120148.pdf', '_receita_jan2025-consol_cer16400_2271_27023424.pdf', '_receita_jul2025-consolidado_04031147.pdf', '_receita_jun2025-consolidado_cer16400_1830_25122119.pdf', '_receita_mai2025-consolidado_cer16400_1475_25012608.pdf', '_receita_nov2025-consol_cer16400_1729_28115912.pdf', '_receita_out2025-consolidada_24113558.pdf', '_receita_set2025-consol_cer16400_4856_12040708.pdf']


- Loop que vai percorrer os pdfs

In [8]:
df_total = pd.DataFrame()

for arquivo in arquivos:
    caminho = os.path.join(pasta, arquivo)

    with pdfplumber.open(caminho) as pdf:
        texto = ""

        for pagina in pdf.pages:
            pagina_texto = pagina.extract_text()
            if pagina_texto:
                texto += pagina_texto + "\n"

    matches = re.findall(padrao, texto)

    dados = []

    for m in matches:
        valores = re.findall(r"[\d\.,\-]+", m[2])

        dados.append({
            "arquivo": arquivo, 
            "codigo": m[0],
            "descricao": m[1],
            "mes": valores[1]
        })

    df_mes = pd.DataFrame(dados)
    df_total = pd.concat([df_total, df_mes], ignore_index=True)

- Extraindo com regex

In [9]:
matches = re.findall(padrao, texto)

dados = []

for m in matches:
        valores = re.findall(r"[\d\.,\-]+", m[2])

        dados.append({
            "arquivo": arquivo, 
            "codigo": m[0],
            "descricao": re.split(r"Anterior|Arrecadado|Mês", m[1])[0].strip(),
            "anterior": valores[0],
            "mes": valores[1],
            "ano": valores[2],
            "previsto": valores[3],
            "atualizado": valores[4],
            "diferenca": valores[5],
        })

- Criando data frame do mes e juntando tudo

In [10]:
mapa_meses = {
    "jan": "janeiro",
    "fev": "fevereiro",
    "mar": "março",
    "abr": "abril",
    "mai": "maio",
    "jun": "junho",
    "jul": "julho",
    "ago": "agosto",
    "set": "setembro",
    "out": "outubro",
    "nov": "novembro",
    "dez": "dezembro"
}

- criando colunas de mes

In [11]:
df_total["mes_ref"] = df_total["arquivo"].str.lower().apply(
    lambda x: next((v for k, v in mapa_meses.items() if k in x), None))

-limpeza

In [12]:
df_total = df_total.rename(columns={"mes": "valor"})
df_total["valor"] = df_total["valor"].apply(converter)
df_total.head()


,arquivo,codigo,descricao,valor,mes_ref
0,_receita_ago2025-consol_cer16400_9426_29094402...,1.1.1.2.50.0.1.001,IMPOSTOS S/PREDIAL URBANO\nAnterior A r r e c ...,819217.92,agosto
1,_receita_ago2025-consol_cer16400_9426_29094402...,1.1.1.2.50.0.1.002,IMP. S/PROP. TERRITORIAL URBANA\nAnterior A r ...,508681.64,agosto
2,_receita_ago2025-consol_cer16400_9426_29094402...,1.1.1.2.50.0.2.001,IPTU M/J\nAnterior A r r e c a d a d o Mês A r...,6024.22,agosto
3,_receita_ago2025-consol_cer16400_9426_29094402...,1.1.1.2.50.0.3.001,IPTU DIV ATIVA\nAnterior A r r e c a d a d o M...,1216196.24,agosto
4,_receita_ago2025-consol_cer16400_9426_29094402...,1.1.1.2.50.0.4.001,M/J DIV ATIVA IPTU\nAnterior A r r e c a d a d...,572379.92,agosto


- limpando descrição

In [13]:
def limpar_texto(txt):
    txt = str(txt)
    txt = re.split(r"Anterior|Arrecadado|Mês", txt)[0]
    txt = txt.replace("\n", " ")
    txt = re.sub(r"\s+", " ", txt)
    return txt.strip()

df_total["descricao"] = df_total["descricao"].apply(limpar_texto)

- dataset final

In [14]:
df_final = df_total[["mes_ref", "codigo", "descricao", "valor"]]
df_final

,mes_ref,codigo,descricao,valor
0,agosto,1.1.1.2.50.0.1.001,IMPOSTOS S/PREDIAL URBANO,819217.92
1,agosto,1.1.1.2.50.0.1.002,IMP. S/PROP. TERRITORIAL URBANA,508681.64
2,agosto,1.1.1.2.50.0.2.001,IPTU M/J,6024.22
3,agosto,1.1.1.2.50.0.3.001,IPTU DIV ATIVA,1216196.24
4,agosto,1.1.1.2.50.0.4.001,M/J DIV ATIVA IPTU,572379.92
...,...,...,...,...
3001,setembro,7.9.2.3.99.0.1.001,Outros Ressarcimentos - Daemo,55549.32
3002,setembro,7.9.2.3.99.0.1.002,Outros Ressarcimentos - Instituto,5164.89
3003,setembro,7.9.2.3.99.0.1.003,Outros Ressarcimentos - Câmara,6334.98
3004,setembro,7.9.9.9.01.0.1.001,APORTES PARA COBERTURA DE DEFICIT,40476.20


- Ordenando meses

In [19]:
ordem_meses = [
    "janeiro", "fevereiro", "março", "abril",
    "maio", "junho", "julho", "agosto",
    "setembro", "outubro", "novembro", "dezembro"
]

df_final["mes_ref"] = pd.Categorical(
    df_final["mes_ref"],
    categories=ordem_meses,
    ordered=True
)

df_final = df_final.sort_values("mes_ref")


In [20]:
df_final.groupby("mes_ref")["valor"].sum()

mes_ref
janeiro      31777932.59
fevereiro    35856438.90
março        40524651.38
abril        39188060.05
maio         31825427.28
junho        30371449.32
julho        33114712.72
agosto       40585069.07
setembro     31124867.31
outubro      39750778.22
novembro     30952798.56
dezembro     42952092.67
Name: valor, dtype: float64

In [21]:
df_final.to_csv("Dados_Limpos/receita_limpa.csv", index=False)
